# Silver Sales Layer
Cleans and loads all `sales_data.bronze` tables into `sales_data.silver`, applying data quality checks for **duplicates**, **null values**, and **special characters** in string columns.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "sales_data"
SILVER = f"{CATALOG}.silver"

# Table specs: primary key + Y/N flag columns to cast to boolean in silver
TABLE_SPECS = {
    "customers":   {"pk": "customer_id", "boolean_cols": ["is_active"]},
    "products":    {"pk": "product_id",  "boolean_cols": ["is_active"]},
    "stores":      {"pk": "store_id",    "boolean_cols": []},
    "promotions":  {"pk": "promo_id",    "boolean_cols": []},
    "orders":      {"pk": "order_id",    "boolean_cols": []},
    "order_items": {"pk": "item_id",     "boolean_cols": []},
    "returns":     {"pk": "return_id",   "boolean_cols": []},
}

# Characters allowed in string columns (covers '+1-555-...' phones, 'Store #1', emails, etc.)
ALLOWED_PATTERN = "[^A-Za-z0-9 @.\\-_&'()/,+#]"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
print(f"Schema ready: {SILVER}")

In [0]:
def audit_table(table):
    """Data quality audit of one bronze table: duplicates, nulls, special characters."""
    pk = TABLE_SPECS[table]["pk"]
    df = spark.table(f"{CATALOG}.bronze.{table}")
    total = df.count()
    dup_rows = total - df.select(pk).distinct().count()
    null_pk = df.filter(F.col(pk).isNull()).count()

    null_counts = df.agg(*[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns if not c.startswith("_")
    ]).first().asDict()
    nulls_found = {c: n for c, n in null_counts.items() if n} or None

    string_cols = [f.name for f in df.schema.fields
                   if f.dataType.simpleString() == "string" and not f.name.startswith("_")]
    specials_found = {c: n for c, n in {
        c: df.filter(F.col(c).rlike(ALLOWED_PATTERN)).count() for c in string_cols
    }.items() if n} or None

    return {"table": table, "rows": total, "duplicate_rows": dup_rows, "null_pk_rows": null_pk,
            "columns_with_nulls": str(nulls_found), "columns_with_special_chars": str(specials_found)}

audit_rows = [audit_table(t) for t in TABLE_SPECS]
display(spark.createDataFrame(audit_rows))

In [0]:
def clean_and_load(table):
    """Clean one bronze table and load it to silver."""
    spec = TABLE_SPECS[table]
    pk, bool_cols = spec["pk"], spec["boolean_cols"]
    df = spark.table(f"{CATALOG}.bronze.{table}")

    # 1) Trim strings and strip characters outside the allowed whitelist
    string_cols = [f.name for f in df.schema.fields
                   if f.dataType.simpleString() == "string" and not f.name.startswith("_")]
    for c in string_cols:
        df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), ALLOWED_PATTERN, "")))

    # 2) Cast Y/N flag columns to boolean
    for c in bool_cols:
        df = df.withColumn(c, F.when(F.upper(F.col(c)) == "Y", True)
                                 .when(F.upper(F.col(c)) == "N", False))

    # 3) Drop rows with a null primary key
    df = df.filter(F.col(pk).isNotNull())

    # 4) Deduplicate on the primary key, keeping the most recently ingested row
    w = Window.partitionBy(pk).orderBy(F.col("_ingested_at").desc())
    df = (df.withColumn("_rn", F.row_number().over(w))
            .filter(F.col("_rn") == 1)
            .drop("_rn", "_rescued_data")
            .withColumn("_processed_at", F.current_timestamp()))

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER}.{table}")
    return df.count()

for t in TABLE_SPECS:
    n = clean_and_load(t)
    print(f"{SILVER}.{t}: wrote {n:,} rows")

In [0]:
verify_rows = []
for t in TABLE_SPECS:
    pk = TABLE_SPECS[t]["pk"]
    sdf = spark.table(f"{SILVER}.{t}")
    silver_count = sdf.count()
    dup_pk = silver_count - sdf.select(pk).distinct().count()
    null_pk = sdf.filter(F.col(pk).isNull()).count()
    string_cols = [f.name for f in sdf.schema.fields if f.dataType.simpleString() == "string"]
    special_cond = F.lit(False)
    for c in string_cols:
        special_cond = special_cond | F.col(c).rlike(ALLOWED_PATTERN)
    specials = sdf.filter(special_cond).count() if string_cols else 0
    verify_rows.append({"table": t, "silver_rows": silver_count, "duplicate_pk": dup_pk,
                        "null_pk": null_pk, "rows_with_special_chars": specials})

display(spark.createDataFrame(verify_rows))